# Preparacion reproducible en Colab
Seleccionar runtime GPU T4 para entrenamiento/inferencia. Repositorio privado: cargar un ZIP autorizado mediante selector; no escribir tokens en celdas. Se trabaja desde raiz del proyecto. Las salidas vacias no constituyen evidencia de ejecucion.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith('.zip'))
with zipfile.ZipFile(archive) as z:
    z.extractall('/content/proyecto')
roots = list(Path('/content/proyecto').rglob('requirements.txt'))
assert len(roots) == 1, 'El ZIP debe contener una sola copia del proyecto'
os.chdir(roots[0].parent)
print(Path.cwd())

In [ ]:
%pip install torch==2.6.0 --index-url https://download.pytorch.org/whl/cu124
%pip install -r requirements.txt
!python -m pip check

In [ ]:
import torch, transformers, peft, accelerate, bitsandbytes, datasets
print('torch', torch.__version__, 'CUDA', torch.cuda.is_available())
print('transformers', transformers.__version__, 'PEFT', peft.__version__)
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

## Antes de entrenar
Completar notebook 01: al menos 2000 pares aceptados por MySQL real; completar baseline del notebook 03. Leer docs/hipotesis_experimentos.md, escrito previamente. No usar esquemas de validacion para entrenamiento. Los procesos separados liberan GPU entre A y B.

In [ ]:
from src.training.pipeline import load_records
records = load_records('datos/entrenamiento/aceptados.jsonl')
print('Ejemplos con evidencia:', len(records))
assert torch.cuda.is_available(), 'Seleccionar runtime GPU T4'
print(Path('docs/hipotesis_experimentos.md').read_text())

In [ ]:
!python -m scripts.train --config configs/lora_a.json

In [ ]:
!python -m scripts.train --config configs/lora_b.json

In [ ]:
import json
for name in ('a','b'):
    path = Path(f'adapters_experimentos/config_{name}/run.json')
    run = json.loads(path.read_text())
    assert run['estado'] == 'completado', run
    print(name, run)
    assert Path(f'adapters_experimentos/config_{name}/adapter_model.safetensors').exists()

Descargar adapters_experimentos y resultados/curvas_entrenamiento antes de cerrar runtime; contienen pesos, configuracion, tiempos, VRAM, hashes y curvas reales. Evaluar ambos antes de promover un adapter a adapter/.

In [ ]:
import shutil
shutil.make_archive('/content/entrenamientos', 'zip', '.', 'adapters_experimentos')
shutil.make_archive('/content/curvas', 'zip', '.', 'resultados/curvas_entrenamiento')
files.download('/content/entrenamientos.zip')
files.download('/content/curvas.zip')